Given a 2D character matrix grid, where grid[i][j] is either 'X', 'Y', or '.', return the number of submatrices that contain:

grid[0][0]
an equal frequency of 'X' and 'Y'.
at least one 'X'.
 

Example 1:

Input: grid = [["X","Y","."],["Y",".","."]]

Output: 3

Explanation:



Example 2:

Input: grid = [["X","X"],["X","Y"]]

Output: 0

Explanation:

No submatrix has an equal frequency of 'X' and 'Y'.

Example 3:

Input: grid = [[".","."],[".","."]]

Output: 0

Explanation:

No submatrix has at least one 'X'.

 

Constraints:

1 <= grid.length, grid[i].length <= 1000
grid[i][j] is either 'X', 'Y', or '.'.

In [ ]:
# this nethod is based on the finding the submatrix with the sum 0.
# and check 
class Solution:
    def numberOfSubmatrices(self, grid: list[list[str]]) -> int:
        # first will create a prefix sum matrix.
        m, n = len(grid), len(grid[0])
        psum = [[0 for _ in range(n+1)] for _ in range(m+1)] 
        has_x = [[False for _ in range(n+1)] for _ in range(m+1)] 

        def get_val(val):
            if val == "X":
                return 1
            if val == "Y":
                return -1
            return 0
        
        # find the psum:
        for i in range(1, m+1):
            for j in range(1, n+1):
                psum[i][j] = get_val(grid[i-1][j-1]) + psum[i][j-1] + psum[i-1][j] - psum[i -1][j-1] 
        
        # find how many submatrix has x.
        for i in range(1, m+1):
            for j in range(1, n+1):
                has_x[i][j] = (grid[i-1][j-1] == "X") or (has_x[i-1][j] == True) or (has_x[i][j-1] == True) 
        

        # get the count.
        count = 0
        for i in range(1, m+1):
            for j in range(1, n+1):
                if psum[i][j] == 0 and has_x[i][j] == True:
                    count +=1
        return count 
    

# tc - O(n*m)
# sc - O(n*m) + O(n*m)

In [12]:
Solution().numberOfSubmatrices([["X","Y","."],["Y",".","."]])

3

In [13]:
Solution().numberOfSubmatrices(grid = [["X","X"],["X","Y"]])

0

In [14]:
Solution().numberOfSubmatrices(grid = [[".","."],[".","."]])

0

In [ ]:
# we need to reduce hte space,
# we keep the running count of X and y row wise.
# then find the results.


class Solution:
    def numberOfSubmatrices(self, grid: list[list[str]]) -> int:
        rows = len(grid)
        cols = len(grid[0])
        sumX = [0] * cols
        sumY = [0] * cols
        res = 0

        for i in range(rows):
            rx = 0
            ry = 0
            for j in range(cols):
                if grid[i][j] == 'X':
                    rx += 1
                elif grid[i][j] == 'Y':
                    ry += 1
                
                sumX[j] += rx
                sumY[j] += ry
                
                if sumX[j] > 0 and sumX[j] == sumY[j]:
                    res += 1

        return res
    
# tc - O(n*m)
# sc - O(m)

## Approach 2: Space-Optimized O(cols)

**Key insight:** Every valid submatrix must start at `grid[0][0]`.  
So we only count submatrices ending at each `(i, j)`.

**Idea:**  
- `rx`, `ry` = running X/Y count within current row (reset each row)  
- `sumX[j]`, `sumY[j]` = cumulative X/Y count in rectangle `(0,0) → (i,j)` (carry across rows)  
- Valid if `sumX[j] == sumY[j] > 0`

---

### Example — 4×5 grid:

```
     j=0  j=1  j=2  j=3  j=4
i=0 [ X    Y    .    X    Y  ]
i=1 [ .    .    Y    X    .  ]
i=2 [ Y    X    .    .    X  ]
i=3 [ X    .    X    Y    .  ]
```

---

**Row 0:** `X  Y  .  X  Y`  — rx/ry accumulate left→right

```
j:       0    1    2    3    4
rx:      1    1    1    2    2
ry:      0    1    1    1    2
sumX:  [ 1    1    1    2    2 ]
sumY:  [ 0    1    1    1    2 ]
valid:       ✓    ✓              ✓   → 3 hits
```

---

**Row 1:** `Y  .  Y  X  .`  — rx/ry reset to 0, sumX/sumY carry forward

```
j:       0    1    2    3    4
rx:      0    0    0    1    1
ry:      1    1    2    2    2
sumX:  [ 1    1    1    3    3 ]   ← sumX[j] += rx
sumY:  [ 1    2    3    3    4 ]   ← sumY[j] += ry
valid:  ✓                          → 1 hit
```
*(sumX[0]=1+0=1, sumY[0]=0+1=1 → rectangle from (0,0)→(1,0) has 1X, 1Y)*

---

**Row 2:** `Y  X  .  .  X`  — rx/ry reset to 0

```
j:       0    1    2    3    4
rx:      0    1    1    1    2
ry:      1    1    1    1    1
sumX:  [ 1    2    2    4    5 ]
sumY:  [ 2    3    4    4    5 ]
valid:                          ✓   → 1 hit
```

---

**Row 3:** `X  .  X  Y  .`  — rx/ry reset to 0

```
j:       0    1    2    3    4
rx:      1    1    2    2    2
ry:      0    0    0    1    1
sumX:  [ 2    3    4    6    7 ]
sumY:  [ 2    3    4    5    6 ]
valid:  ✓    ✓    ✓             → 3 hits
```

---

**Total = 3 + 1 + 1 + 3 = 8**